# Installing packages

In [54]:
!pip install torch torchtext sentencepiece nltk bert-score pandas numpy matplotlib

# Loading Dataset

In [55]:
import pandas as pd

def dataset_load(sa_path, en_path):
    sa_df = pd.read_csv(sa_path)
    en_df = pd.read_csv(en_path)
    merged_df = pd.merge(sa_df, en_df, on="Source_id")
    merged_df.rename(columns={"Sentence_sa": "source_lang", "Sentence_en": "target_lang"}, inplace=True)
    return merged_df[["Source_id", "source_lang", "target_lang"]]

train_df = dataset_load("data/train_sa_10000.csv", "data/train_en_10000.csv")
dev_df = dataset_load("data/dev_sa_1000.csv", "data/dev_en_1000.csv")
test_df = dataset_load("data/test_sa_1000.csv", "data/test_en_1000.csv")


# SentencePiece Tokenizer training

In [56]:
import sentencepiece as sent_pi

def train_sentence_piece(sentences, model_prefix, vocab_size=8000):
  with open(f"{model_prefix}.txt", "w", encoding="utf-8") as f:
    for sentence in sentences:
      f.write(sentence + "\n")

  sent_pi.SentencePieceTrainer.train(
    input=f"{model_prefix}.txt",
    model_prefix=model_prefix,
    vocab_size=vocab_size,
    character_coverage=1.0,
    model_type="bpe",
  )

## Preparing Sentences


In [57]:
source_sentences = train_df["source_lang"].tolist()
target_sentences = train_df["target_lang"].tolist()

train_sentence_piece(source_sentences, "spm_source", vocab_size=8000)
train_sentence_piece(target_sentences, "spm_target", vocab_size=8000)

## Load Tokenizers

In [58]:
source_sp = sent_pi.SentencePieceProcessor()
source_sp.load("spm_source.model")
target_sp = sent_pi.SentencePieceProcessor()
target_sp.load("spm_target.model")

True

# Dataset Loader and Dataset

## Importing packages

In [59]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

## Translation Dataset

In [60]:
class TranslationDataset(Dataset):
    def __init__(self, data_frame, source_sp, target_sp, max_length=128):
        self.data_frame = data_frame
        self.source_sp = source_sp
        self.target_sp = target_sp
        self.max_length = max_length
        self.pad_index = 0

    def __len__(self):
        return len(self.data_frame)

    def __getitem__(self, index):
        source_text = self.data_frame.iloc[index]["source_lang"]
        target_text = self.data_frame.iloc[index]["target_lang"]
        source_indexes = [self.source_sp.bos_id()] + self.source_sp.EncodeAsIds(source_text) + [self.source_sp.eos_id()]
        target_indexes = [self.target_sp.bos_id()] + self.target_sp.EncodeAsIds(target_text) + [self.target_sp.eos_id()]
        source_indexes = source_indexes[:self.max_length]
        target_indexes = target_indexes[:self.max_length]
        return torch.tensor(source_indexes, dtype=torch.long), torch.tensor(target_indexes, dtype=torch.long)

## Creating a collate function

In [61]:
def collate(batch):
  source_batch, target_batch = zip(*batch)
  source_lengths = [len(s) for s in source_batch]
  target_lengths = [len(t) for t in target_batch]
  source_padded = pad_sequence(source_batch, batch_first=True, padding_value=0)
  target_padded = pad_sequence(target_batch, batch_first=True, padding_value=0)
  return source_padded, source_lengths, target_padded, target_lengths

## Using TranslationDataset Class

In [62]:
train_dataset = TranslationDataset(train_df, source_sp, target_sp)
dev_dataset = TranslationDataset(dev_df, source_sp, target_sp)
test_dataset = TranslationDataset(test_df, source_sp, target_sp)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate, num_workers=4, pin_memory=True)
dev_loader = DataLoader(dev_dataset, batch_size=32, shuffle=False, collate_fn=collate, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate, num_workers=2, pin_memory=True)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


# Creating a Transformer Model
Here we will create transformer model. We will be using following details for this purpose:-


*   Embedding Layer
*   Positional Encodeing
*   Multi-head Attention
*   Feed-forward networks
*   Encoder and Decoder Stacks
*   Final linear layer with softmax




## Positional Encoding

We will be using Sinusoidal positional encoding. It is a technique to inject information about sequence order of tokens

### Importing packages

In [63]:
import math
import torch.nn as neural_net

### Creating class for Positional Encoder

In [64]:
class PositionalEncoding(neural_net.Module):
  def __init__(self, d_model, max_length=5000):
    super().__init__()
    positonal_encoder = torch.zeros(max_length, d_model)
    position = torch.arange(0, max_length, dtype=torch.float).unsqueeze(1)
    divison_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    positonal_encoder[:, 0::2] = torch.sin(position * divison_term)
    positonal_encoder[:, 1::2] = torch.cos(position * divison_term)
    positonal_encoder = positonal_encoder.unsqueeze(0)
    self.register_buffer("positonal_encoder", positonal_encoder)

  def forward(self, x):
    x = x + self.positonal_encoder[:, :x.size(1)]
    return x


## Multi Head Attention and Transformer Block

### Transformer Encoder Layer

In [65]:
class TransformerEncoderLayer(neural_net.Module):
  def __init__(self, d_model, nhead, feedforward_dimension=2048, dropout=0.1):
    super().__init__()
    self.self_attention = neural_net.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
    self.linear1 = neural_net.Linear(d_model, feedforward_dimension)
    self.linear2 = neural_net.Linear(feedforward_dimension, d_model)
    self.norm1 = neural_net.LayerNorm(d_model)
    self.norm2 = neural_net.LayerNorm(d_model)
    self.dropout = neural_net.Dropout(dropout)
    self.activation_relu = neural_net.ReLU()

  def forward(self, source, source_mask=None, source_key_padding_mask=None):
    attention_out, _ = self.self_attention(source, source, source, attn_mask=source_mask, key_padding_mask=source_key_padding_mask)
    source = source + self.dropout(attention_out)
    source = self.norm1(source)

    feedforward_out = self.linear2(self.dropout(self.activation_relu(self.linear1(source))))
    source = source + self.dropout(feedforward_out)
    source = self.norm2(source)
    return source


### Transformer Decoder Layer

In [66]:
class TransformerDecoderLayer(neural_net.Module):
  def __init__(self, d_model, nhead, feedforward_dimension=2048, dropout=0.1):
    super().__init__()
    self.self_attention = neural_net.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
    self.cross_attention = neural_net.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
    self.linear1 = neural_net.Linear(d_model, feedforward_dimension)
    self.linear2 = neural_net.Linear(feedforward_dimension, d_model)
    self.norm1 = neural_net.LayerNorm(d_model)
    self.norm2 = neural_net.LayerNorm(d_model)
    self.norm3 = neural_net.LayerNorm(d_model)
    self.dropout = neural_net.Dropout(dropout)
    self.activation_relu = neural_net.ReLU()

  def forward(self, target, memory, target_mask=None, target_key_padding_mask=None, memory_key_padding_mask=None):
    attention_out, _ = self.self_attention(target, target, target, attn_mask=target_mask, key_padding_mask=target_key_padding_mask)
    target = target + self.dropout(attention_out)
    target = self.norm1(target)

    cross_atten_out, _ = self.cross_attention(target, memory, memory, key_padding_mask=memory_key_padding_mask)
    target = target + self.dropout(cross_atten_out)
    target = self.norm2(target)

    feedforward_out = self.linear2(self.dropout(self.activation_relu(self.linear1(target))))
    target = target + self.dropout(feedforward_out)
    target = self.norm3(target)
    return target

## Encoder and Decoder Stack

### Encoder Transformer

In [67]:
class TransformerEncoder(neural_net.Module):
  def __init__(self, vocabulary_size, d_model, nhead, number_of_layer, forward_dimension=2048, dropout=0.1, max_length=5000):
    super().__init__()
    self.embedding = neural_net.Embedding(vocabulary_size, d_model, padding_idx=0)
    self.positional_encoding = PositionalEncoding(d_model, max_length)
    self.layers = neural_net.ModuleList([
        TransformerEncoderLayer(d_model, nhead, feedforward_dimension= forward_dimension, dropout=dropout)
        for _ in range(number_of_layer)
    ])
    self.dropout = neural_net.Dropout(dropout)
    self.d_model = d_model

  def forward(self, source, source_mask=None, source_key_padding_mask=None):
    source = self.embedding(source) * math.sqrt(self.d_model)
    source = self.positional_encoding(source)
    source = self.dropout(source)
    for layer in self.layers:
      source = layer(source, source_mask, source_key_padding_mask)
    return source

### Decoder Transformer

In [68]:
class TransformerDecoder(neural_net.Module):
  def __init__(self, vocabulary_size, d_model, nhead, number_of_layer, forward_dimension=2048, dropout=0.1, max_length=5000):
    super().__init__()
    self.embedding = neural_net.Embedding(vocabulary_size, d_model, padding_idx=0)
    self.positional_encoding = PositionalEncoding(d_model, max_length)
    self.layers = neural_net.ModuleList([
        TransformerDecoderLayer(d_model, nhead, feedforward_dimension= forward_dimension, dropout=dropout)
        for _ in range(number_of_layer)
    ])
    self.linear = neural_net.Linear(d_model, vocabulary_size)
    self.dropout = neural_net.Dropout(dropout)
    self.d_model = d_model

  def forward(self, target, memory, target_mask=None, target_key_padding_mask=None, memory_key_padding_mask=None):
    target = self.embedding(target) * math.sqrt(self.d_model)
    target = self.positional_encoding(target)
    target = self.dropout(target)
    for layer in self.layers:
      target = layer(target, memory, target_mask, target_key_padding_mask, memory_key_padding_mask)
    return self.linear(target)

## Sequence to Sequence Model (Seq2Seq Model)

In [69]:
class TransformerSeq2SeqModel(neural_net.Module):
  def __init__(self, source_vocab_size, target_vocab_size, d_model=512, nhead=8, number_of_encoder_layer=6, number_of_decoder_layer=6, forward_dimension=2048, dropout=0.1, max_length=5000):
    super().__init__()
    self.encoder = TransformerEncoder(source_vocab_size, d_model, nhead, number_of_encoder_layer, forward_dimension, dropout, max_length)
    self.decoder = TransformerDecoder(target_vocab_size, d_model, nhead, number_of_decoder_layer, forward_dimension, dropout, max_length)
    self.target_vocab_size = target_vocab_size
    self.d_model = d_model


  def forward(self, source, target, source_mask=None, target_mask=None, source_padding_mask=None, target_padding_mask=None, memory_padding_mask=None):
    memory = self.encoder(source, source_mask, source_padding_mask)
    output = self.decoder(target, memory, target_mask, target_padding_mask, memory_padding_mask)
    return output

  def generate_square_subsequent_mask(self, size):
    mask = torch.triu(torch.ones(size, size), diagonal=1).bool()
    return mask

# Training Loop

## Importing packages

In [70]:
import torch.optim as optim
from torch.nn.utils import clip_grad_norm_

### Training epochs

In [71]:
def count_parameters(model):
  return sum(param.numel() for param in model.parameters() if param.requires_grad)

In [72]:
def train_epoch(model, loader, optimizer, criterion, device, schedular=None):
  model.train()
  total_loss = 0
  for source, source_length, target, target_lengths in loader:
    optimizer.zero_grad()
    source = source.to(device)
    target = target.to(device)
    target_input = target[:, :-1]
    target_output = target[:, 1:]

    source_mask = None
    target_mask = model.generate_square_subsequent_mask(target_input.size(1)).to(device)
    source_padding_mask = (source == 0)
    target_padding_mask = (target_input == 0)
    optimizer.zero_grad()
    logits = model(
        source,
        target_input,
        source_mask = source_mask,
        target_mask = target_mask,
        source_padding_mask = source_padding_mask,
        target_padding_mask = target_padding_mask,
        memory_padding_mask = source_padding_mask
    )
    loss = criterion(logits.reshape(-1, logits.size(-1)), target_output.reshape(-1))
    loss.backward()
    clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    if schedular:
      schedular.step()
    total_loss += loss.item()
  return total_loss / len(loader)

# Beam Search Implementation

## Creating translate beam function

In [73]:
def translate_beam(model, source, target_token_sp, beam_size=4, max_length=128, device="cuda"):
  model.eval()
  batch_size = source.size(0)
  source_padding_mask = (source == 0)
  memory = model.encoder(source, source_key_padding_mask=source_padding_mask)
  bos_index = target_token_sp.bos_id()
  eos_index = target_token_sp.eos_id()

  sequences = torch.full((batch_size, beam_size, 1), bos_index, dtype=torch.long, device=device)
  scores = torch.zeros(batch_size, beam_size, device=device)

  for step in range(max_length):
    flat_sequences = sequences.view(batch_size * beam_size, -1)
    target_mask = model.generate_square_subsequent_mask(flat_sequences.size(1)).to(device)
    target_padding_mask = (flat_sequences == 0)
    expanded_memory = memory.repeat_interleave(beam_size, dim=0)
    expanded_source_padding_mask = source_padding_mask.repeat_interleave(beam_size, dim=0)
    logits = model.decoder(
        flat_sequences,
        expanded_memory,
        target_mask=target_mask,
        target_key_padding_mask=target_padding_mask,
        memory_key_padding_mask=expanded_source_padding_mask
    )
    next_logits = logits[:, -1, :]
    next_probs = torch.log_softmax(next_logits, dim=-1)
    next_probs = next_probs.view(batch_size, beam_size, -1)
    combined_scores = scores.unsqueeze(2) + next_probs
    combined_scores = combined_scores.view(batch_size, -1)
    top_scores, top_indices = torch.topk(combined_scores, beam_size, dim=-1)
    beam_indices = top_indices // (target_token_sp.vocab_size())
    token_ids = top_indices % (target_token_sp.vocab_size())

    new_sequences = []
    for b in range(batch_size):
      batch_sequences = []
      for k in range(beam_size):
        previous_beam = beam_indices[b, k].item()
        token = token_ids[b, k].item()
        previous_sequence = sequences[b, previous_beam]
        new_sequence = torch.cat([previous_sequence, torch.tensor([token], device=device)])
        batch_sequences.append(new_sequence)
      new_sequences.append(torch.stack(batch_sequences))

    sequences = torch.stack(new_sequences)
    scores = top_scores

  best_sequence_indices = scores.argmax(dim=-1)
  best_sequences = sequences[torch.arange(batch_size), best_sequence_indices]
  return best_sequences

# Validation And Evaluation

## Validation

In [74]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from bert_score import score as bert_score

In [75]:
def translate_batch(model, source_tokens, source_lengths, target_sp, max_length=128, beam_size=4, device="cuda"):
  model.eval()
  with torch.no_grad():
    source = source_tokens.to(device)
    source_padding_mask = (source == 0)
    memory = model.encoder(source, source_key_padding_mask=source_padding_mask)
    batch_size = source.size(0)
    target_tokens = torch.full((batch_size, 1), target_sp.bos_id(), dtype=torch.long, device=device)
    for _ in range(max_length):
      target_mask = model.generate_square_subsequent_mask(target_tokens.size(1)).to(device)
      target_key_padding_mask = (target_tokens == 0)
      logits = model.decoder(
          target_tokens,
          memory,
          target_mask=target_mask,
          target_key_padding_mask=target_key_padding_mask,
          memory_key_padding_mask=source_padding_mask
        )
      next_token_logits = logits[:, -1, :].argmax(dim=-1, keepdim=True)
      target_tokens = torch.cat([target_tokens, next_token_logits], dim=1)
      if (next_token_logits == target_sp.eos_id()).all():
        break
    return target_tokens


## Evaluation

In [76]:
def evaluate(model, loader, target_token_sp, device, beam_size=4):
  model.eval()
  refs = []
  hyps = []
  source_ids = []
  with torch.no_grad():
    for batch in loader:
      source, source_lengths, target, target_lengths = batch
      source = source.to(device)
      translations = translate_beam(model, source, target_token_sp, beam_size=beam_size, max_length=128, device=device)
      for i in range(source.size(0)):
        source_id = loader.dataset.data_frame.iloc[i]["Source_id"]
        source_ids.append(source_id)
        reference_text = ' '.join(target_token_sp.DecodeIds(target[i].tolist()))
        translation_text = ' '.join(target_token_sp.DecodeIds(translations[i].tolist()))
        refs.append(reference_text)
        hyps.append(translation_text)

  bleu_scores = []
  smooth = SmoothingFunction().method1
  for ref, hype in zip(refs, hyps):
    bleu_score = sentence_bleu([ref.split()], hype.split(), smoothing_function=smooth)
    bleu_scores.append(bleu_score)
  avg_bleu_score = sum(bleu_scores) / len(bleu_scores) if bleu_scores else 0.0
  print(f"Average BLEU Score: {avg_bleu_score}")
  P, R, F1 = bert_score(hyps, refs, lang="en", rescale_with_baseline=True, verbose=True)
  print(f"P: {P.mean().item()}, R: {R.mean().item()}, F1: {F1.mean().item()}")
  avg_bert_f1 = F1.mean().item()

  return avg_bleu_score, avg_bert_f1, source_ids, refs, hyps

# Training the model

A training loop will be set with early stop based on dev BLEU

In [77]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
source_vocab_size = source_sp.vocab_size()
target_vocab_size = target_sp.vocab_size()
model = TransformerSeq2SeqModel(
    source_vocab_size,
    target_vocab_size,
    d_model=512, nhead=8,
    number_of_encoder_layer=6,
    number_of_decoder_layer=6,
    forward_dimension=2048, dropout=0.1,
    max_length=5000
  ).to(device)
print(f"Total parameters to train: {count_parameters(model)}")
criterion = neural_net.CrossEntropyLoss(ignore_index=0, label_smoothing=0.1)
optimizer = optim.Adam(model.parameters(), lr=0.001, betas=(0.9, 0.98), eps=1e-9)


Total parameters to train: 56434496


### Noam Schedular

In [78]:
warmup_steps = 4000
d_model = 512
def noam_scheduler(step):
  step = max(1, step)
  return d_model ** (-0.5) * min(step ** (-0.5), step * warmup_steps ** (-1.5))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=noam_scheduler)

### Training Loop

In [79]:
best_blue = 0
patience = 5
epochs = 30
for epoch in range(1, epochs+1):
  train_loss = train_epoch(model, train_loader, optimizer, criterion, device, scheduler)
  dev_bleu, dev_bert, _, _, _ = evaluate(model, dev_loader, target_sp, device, beam_size=4)
  print(f"Epoch {epoch}: Train Loss:={train_loss:.4f}, dev BLEU:={dev_bleu:.4f}, dev BERT F1:={dev_bert:.4f}")

  if dev_bleu > best_blue:
    best_blue = dev_bleu
    torch.save(model.state_dict(), "best_model.pt")
    patience = 5
  else:
    patience -= 1
    if patience == 0:
      print("Early stopping")
      break


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


AttributeError: 'TranslationDataset' object has no attribute 'data'

# Evaluation

## Evaluate model on test data

In [ ]:
model.load_state_dict(torch.load("best_model.pt"))
test_bleu, test_bert, test_ids, refs, hyps = evaluate(model, test_loader, target_sp, device, beam_size=4)
print(f"Test BLEU: {test_bleu:.4f}, Test BERTScore F1:{test_bert:.4f}")
submission_dataframe = pd.DataFrame({'Source_id': test_ids, 'Sentence_en': hyps})
submission_dataframe.to_csv("submission.csv", index=False, encoding='utf-8')

## Calculating Inference Time

In [ ]:
import time
start_time = time.time()
_, _, _, _, _ = evaluate(model, test_loader, target_sp, device, beam_size=4)
end_time = time.time()
inference_time = end_time - start_time
print(f"Inference Time for test set: {inference_time:.2f} seconds")